In [0]:
df = spark.table('gizmobox.bronze.py_orders')
display(df)

In [0]:
%sql
SELECT * FROM gizmobox.bronze.v_orders;

1. Pre-process the JSOn String to fix the Data quality Issues

[Documentation](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.regexp_replace.html?highlight=regexp_replace)

In [0]:
from pyspark.sql.functions import col, regexp_replace
df_fixed = (
    df
    .select('value',
            regexp_replace('value','"order_date": (\\d{4}-\\d{2}-\\d{2})', '"order_date": "$1"')
            .alias('value_fixed'))
)

display(df_fixed)

2.Transform JSON String to JSON object

Function [schema_of_JSON](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.schema_of_json.html?highlight=schema_of_json)

Function [from_json](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.from_json.html?highlight=from_json)

In [0]:
from pyspark.sql import functions as F
df_with_schema = (
    df_fixed
    .select(F.schema_of_json(F.col('value_fixed')).alias('schema'))
)

display(df_with_schema.limit(1))

In [0]:
order_schema = '''STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>'''

In [0]:
df_json = (
    df_fixed
    .select(F.from_json(F.col('value_fixed'), order_schema).alias('json_values'))

)
    
display(df_json)

3.Write transformed data to the silver schema

In [0]:
df_json.writeTo('gizmobox.silver.py_orders_fixed').createOrReplace()

In [0]:
%sql
SELECT * FROM gizmobox.silver.py_orders_fixed;